# Assignment 6 — DQN on CartPole-v1


In [1]:
from pathlib import Path
import os

candidate_roots = [
    Path.cwd(),
    Path.cwd() / "dqn",
    Path("/content/dqn"),
]

PROJECT_ROOT = None
for candidate in candidate_roots:
    if (candidate / "requirements.txt").exists():
        PROJECT_ROOT = candidate
        break

if PROJECT_ROOT is None:
    PROJECT_ROOT = Path("/content/dqn")
    PROJECT_ROOT.mkdir(parents=True, exist_ok=True)

PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
(PROJECT_ROOT / "logs").mkdir(parents=True, exist_ok=True)
(PROJECT_ROOT / "checkpoints").mkdir(parents=True, exist_ok=True)

os.chdir(PROJECT_ROOT)
print("Project root:", PROJECT_ROOT.resolve())
print("Logs folder:", (PROJECT_ROOT / "logs").resolve())
print("Checkpoints folder:", (PROJECT_ROOT / "checkpoints").resolve())

Project root: /content/dqn
Logs folder: /content/dqn/logs
Checkpoints folder: /content/dqn/checkpoints


In [2]:
%pip install -r requirements.txt -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 536.7/536.7 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 276.4/276.4 kB 24.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.7/89.7 kB 9.5 MB/s eta 0:00:00


[5 pts] Part 1: Environment Setup
• Online reinforcement learning tasks usually require simulation rather than static
datasets. Therefore the setup is a liNle bit diﬀerent.

- a. Please write a requirements.txt file that installs the necessary additional gymnasium
environment that is compatible with your python and torch version.
- b. Use the gymnasium.make(“CartPole-v1”) to create the environment.
Verify that you can step through the environment and render it (useful to understand
what you’re working with). Please provide a brief explana<on of your setup as part of
a README.

In [3]:
!python env_setup.py

2026-04-14 09:41:11,575 [INFO] env_setup — Starting CartPole-v1 environment verification.
Initial observation: [ 0.0273956  -0.00611216  0.03585979  0.0197368 ]
Initial info: {}
Observation shape: (4,)
Action space: Discrete(2)
Observation space: Box([-4.8               -inf -0.41887903        -inf], [4.8               inf 0.41887903        inf], (4,), float32)
2026-04-14 09:41:11,602 [INFO] env_setup — Initial observation: [ 0.0273956  -0.00611216  0.03585979  0.0197368 ]
2026-04-14 09:41:11,603 [INFO] env_setup — Initial info: {}
2026-04-14 09:41:11,603 [INFO] env_setup — Observation shape: (4,)
2026-04-14 09:41:11,603 [INFO] env_setup — Action space: Discrete(2)
2026-04-14 09:41:11,603 [INFO] env_setup — Observation space: Box([-4.8               -inf -0.41887903        -inf], [4.8               inf 0.41887903        inf], (4,), float32)
error: XDG_RUNTIME_DIR not set in the environment.
ALSA lib confmisc.c:855:(parse_card) cannot find card '0'
ALSA lib conf.c:5178:(_snd_config_eval

[30 pts] Part 2: DQN Implementation
• You will implement a DQN agent from scratch (i.e., no higher-level RL libraries such as
Stable-Baselines3, CleanRL, Rllib). Your implementation should include:

- Neural Network Architecture: the input dimension should match the environment’s
observation space (4 for CartPole). The output dimension is the number of discrete
actions (2 for CartPole). You may use a small multilayer perceptron (MLP). Think — 3
layers with ReLU activation. Document your design decisions (e.g. hidden-layer sizes,
activation functions).
- Q-Network & Target Network: Maintain two networks. A Q-Network for action
selection and a target Network for more stable learning (using a periodic hard
update). Ensure you document how/when you update the target network.
- Replay Buﬀer: Implement an experience replay buﬀer for storing (state,
action, reward, next_state, done, info) tuples. Sample mini-batches
from this buﬀer during training.
- Hyperparameters: Clearly define hyper-parameters in code - learning rate, discount
factor, batch size, replay buﬀer size. Document why you chose them (should closely
follow DQN paper choices).

In [4]:
from dqn_components import DQNConfig, DQNAgent, QNetwork, ReplayBuffer

config = DQNConfig()
print(config)

DQNConfig(state_dim=4, action_dim=2, hidden_dim_1=128, hidden_dim_2=128, learning_rate=0.001, gamma=0.99, batch_size=64, replay_buffer_size=100000, min_replay_size=1000, target_update_every=1000, epsilon_start=1.0, epsilon_end=0.05, epsilon_decay=0.995, seed=42, device='cuda')


[30 pts] Part 3: Data Collection

- Implement a function/method for the agent to select an action (greedy or epsilon-
greedy) given the current state. Step through the environment using
env.step(action). Then, store the resulting transition in the replay buﬀer.
- Write an episode loop that shows how you loop over multiple episodes including:
reseting the environment with env.reset(). Collect transitions until done/
terminated. Track total rewards per episode

In [5]:
!python data_collection.py

2026-04-14 09:41:25,916 [INFO] dqn_components — DQNAgent initialized | device=cuda epsilon_start=1.0000 buffer_size=100000
2026-04-14 09:41:25,917 [INFO] data_collection — Starting data collection for 10 episodes.
2026-04-14 09:41:25,918 [INFO] data_collection — Episode 001 | reward=11.0 length=11 epsilon=0.9950 replay_size=11
Episode 001 | Reward:   11.0 | Length:  11 | Epsilon: 0.9950 | Replay Size:     11
2026-04-14 09:41:25,919 [INFO] data_collection — Episode 002 | reward=18.0 length=18 epsilon=0.9900 replay_size=29
Episode 002 | Reward:   18.0 | Length:  18 | Epsilon: 0.9900 | Replay Size:     29
2026-04-14 09:41:25,919 [INFO] data_collection — Episode 003 | reward=10.0 length=10 epsilon=0.9851 replay_size=39
Episode 003 | Reward:   10.0 | Length:  10 | Epsilon: 0.9851 | Replay Size:     39
2026-04-14 09:41:25,920 [INFO] data_collection — Episode 004 | reward=15.0 length=15 epsilon=0.9801 replay_size=54
Episode 004 | Reward:   15.0 | Length:  15 | Epsilon: 0.9801 | Replay Size:  

[30 pts] Part 4: Training Procedure
- Train for a specified number of episodes or time steps (e.g. 500 - 100 episodes). After
some warm-up steps (so that the replay buﬀer has enough samples), begin sampling
mini-batches to update the Q-Network. Periodically update the target network (e.g.
every N=3 episodes).
- The DQN loss is: reward + discount * max_a’Q_target(s’,a’) - Q_online(s,a))^2
- Use a PyTorch Op<mizer (e.g. Adam) to minimize this loss. Track the loss value to
monitor training progress.
- Log relevant metrics (episode reward, average reward over the last N episodes, loss).
Justify your choice of learning rate, batch size, discount factor.

In [6]:
!python training.py

2026-04-14 09:41:33,080 [INFO] dqn_components — DQNAgent initialized | device=cuda epsilon_start=1.0000 buffer_size=100000
2026-04-14 09:41:33,082 [INFO] training — Training initialized | episodes=500 moving_average_window=20 early_stop=475.0
2026-04-14 09:41:33,091 [INFO] dqn_components — Saved Q-network checkpoint to /content/dqn/checkpoints/training_best_model.pt
2026-04-14 09:41:33,091 [INFO] training — New best moving average 11.00 at episode 1. Checkpoint saved to /content/dqn/checkpoints/training_best_model.pt
Episode 001 | Reward:   11.0 | AvgReward(20):   11.00 | Loss:   0.000000 | Epsilon: 0.9850 | Replay:     11 | BestAvg:   11.00
2026-04-14 09:41:33,092 [INFO] training — Episode 001 | Reward:   11.0 | AvgReward(20):   11.00 | Loss:   0.000000 | Epsilon: 0.9850 | Replay:     11 | BestAvg:   11.00
2026-04-14 09:41:33,094 [INFO] dqn_components — Saved Q-network checkpoint to /content/dqn/checkpoints/training_best_model.pt
2026-04-14 09:41:33,095 [INFO] training — New best movi

[5 pts] Part 5: Evaluation Procedure
- During training, at fixed intervals, evaluate the agent without exploration (pure greedy)
for a fixed number of test episodes. Record the average reward across these evaluations.

In [7]:
!python evaluation.py

2026-04-14 09:44:21,895 [INFO] dqn_components — DQNAgent initialized | device=cuda epsilon_start=1.0000 buffer_size=100000
2026-04-14 09:44:21,897 [INFO] evaluation — Evaluation trainer initialized | episodes=500 eval_interval=25 eval_episodes=10
Episode 001 | Reward:   11.0 | AvgReward(20):   11.00 | Loss:   0.000000 | Epsilon: 0.9850 | Replay:     11
2026-04-14 09:44:21,898 [INFO] evaluation — Episode 001 | Reward:   11.0 | AvgReward(20):   11.00 | Loss:   0.000000 | Epsilon: 0.9850 | Replay:     11
Episode 002 | Reward:   18.0 | AvgReward(20):   14.50 | Loss:   0.000000 | Epsilon: 0.9702 | Replay:     29
2026-04-14 09:44:21,899 [INFO] evaluation — Episode 002 | Reward:   18.0 | AvgReward(20):   14.50 | Loss:   0.000000 | Epsilon: 0.9702 | Replay:     29
Episode 003 | Reward:   10.0 | AvgReward(20):   13.00 | Loss:   0.000000 | Epsilon: 0.9557 | Replay:     39
2026-04-14 09:44:21,900 [INFO] evaluation — Episode 003 | Reward:   10.0 | AvgReward(20):   13.00 | Loss:   0.000000 | Epsilo

In [8]:
import os
import shutil

dqn_path = "/content/dqn"
zip_path = "/content/dqn.zip"

# Check if folder exists
if not os.path.exists(dqn_path):
    raise FileNotFoundError(f"{dqn_path} not found.")

# Remove existing zip if it exists
if os.path.exists(zip_path):
    os.remove(zip_path)

# Create zip
shutil.make_archive("/content/dqn", "zip", dqn_path)

print("dqn.zip created successfully!")

# Download (Colab)
from google.colab import files
files.download(zip_path)

dqn.zip created successfully!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>